# 🎭 Face Swap Live — Google Colab (GPU)

**Passo 0 (obrigatório):** `Runtime → Change runtime type → T4 GPU`

Execute as células em ordem.

In [ ]:
# Célula 1 — Verifica GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    print('❌ GPU nao encontrada!')
    print('Va em Runtime → Change runtime type → T4 GPU e reinicie.')

In [ ]:
# Célula 2 — Instala dependências
!pip install -q flask flask-socketio "pyngrok==6.0.0" eventlet \
    insightface onnxruntime-gpu opencv-python-headless \
    huggingface_hub pillow numpy
print('✅ Instalacao concluida!')

In [ ]:
# Célula 3 — Clona o repositório
import os
if not os.path.exists('faceSwapLive2'):
    !git clone https://github.com/moniquepavan/faceSwapLive2.git
else:
    !cd faceSwapLive2 && git pull
os.chdir('faceSwapLive2')
print('✅ Diretorio:', os.getcwd())

In [ ]:
# Célula 4 — Baixa o modelo inswapper (~280 MB)
import os, shutil
from huggingface_hub import hf_hub_download

DEST = 'models/inswapper_128.onnx'
os.makedirs('models', exist_ok=True)

if os.path.exists(DEST):
    print(f'✅ Modelo ja existe ({os.path.getsize(DEST)/1e6:.0f} MB) — pulando download.')
else:
    token = input('Cole seu token do Hugging Face (huggingface.co/settings/tokens): ').strip()
    print('Baixando...')
    path = hf_hub_download(
        repo_id='hacksider/deep-live-cam',
        filename='inswapper_128_fp16.onnx',
        token=token,
        local_dir='models'
    )
    if os.path.abspath(path) != os.path.abspath(DEST):
        shutil.move(path, DEST)
    print(f'✅ Download concluido! {os.path.getsize(DEST)/1e6:.0f} MB')

In [ ]:
# Célula 5 — Inicia servidor + ngrok
import subprocess, threading, time, os, sys
from pyngrok import ngrok, conf

# (Opcional) token ngrok gratuito para evitar limite de conexões
# Crie conta em ngrok.com e cole o token aqui:
# ngrok.set_auth_token('SEU_TOKEN_NGROK')

# Fecha túneis anteriores se houver
ngrok.kill()

# Inicia servidor em processo separado
server = subprocess.Popen(
    [sys.executable, 'app_colab.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Mostra logs do servidor em background
def print_logs():
    for line in server.stdout:
        print('[servidor]', line.rstrip())
threading.Thread(target=print_logs, daemon=True).start()

print('Aguardando servidor iniciar (5s)...')
time.sleep(5)

# Abre túnel ngrok na porta 5000
tunnel = ngrok.connect(5000, bind_tls=True)
url = tunnel.public_url

print()
print('=' * 60)
print(f'  URL PUBLICA: {url}')
print('=' * 60)
print()
print('1. Abra a URL acima no navegador')
print('2. Aguarde "Pronto! [CUDA GPU]" aparecer no status')
print('3. Faca upload de uma foto e inicie a camera')
print()
print('IMPORTANTE: mantenha esta celula rodando enquanto usar o app.')

In [ ]:
# Célula 6 — Manter sessão ativa (execute em paralelo com célula 5)
# Evita que o Colab encerre por inatividade. Ctrl+C para parar.
import time
print('Mantendo sessao ativa. Ctrl+C para encerrar.')
try:
    i = 0
    while True:
        time.sleep(60)
        i += 1
        print(f'[{i} min] Servidor ativo.')
except KeyboardInterrupt:
    print('Encerrado.')